<a href="https://colab.research.google.com/github/tiwarisubash956/Machine-Learning-Algorithms/blob/main/HyperparameterTuning_using_Optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 20.7 MB/s eta 0:00:00


In [2]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

In [3]:
# load the pipa indian diabetes dataset from repo
# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)


In [4]:
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [5]:
# replace zero value with nan in columns where zero is not avalid values
columns_with_missing_value = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[columns_with_missing_value] = df[columns_with_missing_value].replace(0, np.nan)

df.fillna(df.mean(), inplace=True)

print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [6]:
# split into featues and target
X = df.drop('Outcome', axis=1)
y = df['Outcome']
#split into test train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

# define obejective function
def objective(trial):
    # define hyperparameters to be optimized
    n_estimators = trial.suggest_int('n_estimators', 50,100)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    # min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    # min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

    # create a random forest classifier with the suggested hyperparameters
    clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,random_state=42)

    # perform 3 fold cross validation and caluclate accuracy
    scores = cross_val_score(clf, X_train, y_train, cv=3, scoring='accuracy')

    # return the mean accuracy as the objective value
    return np.mean(scores)


In [10]:
# create a study objective and optimize the objective function
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=100)

# print the best hyperparameters and accuracy
print('Best hyperparameters:', study.best_params)
print('Best accuracy:', study.best_value)

[I 2026-08-16 15:27:55,705] A new study created in memory with name: no-name-6c1f70f2-0e80-4e4d-bbc6-6480461c7fd5
[I 2026-08-16 15:27:56,194] Trial 0 finished with value: 0.7654790371433126 and parameters: {'n_estimators': 81, 'max_depth': 7}. Best is trial 0 with value: 0.7654790371433126.
[I 2026-08-16 15:27:56,722] Trial 1 finished with value: 0.7638211382113821 and parameters: {'n_estimators': 92, 'max_depth': 9}. Best is trial 0 with value: 0.7654790371433126.
[I 2026-08-16 15:27:57,184] Trial 2 finished with value: 0.7654630958074287 and parameters: {'n_estimators': 77, 'max_depth': 9}. Best is trial 0 with value: 0.7654790371433126.
[I 2026-08-16 15:27:57,691] Trial 3 finished with value: 0.7768611509644509 and parameters: {'n_estimators': 92, 'max_depth': 6}. Best is trial 3 with value: 0.7768611509644509.
[I 2026-08-16 15:27:58,019] Trial 4 finished with value: 0.7736091184441256 and parameters: {'n_estimators': 53, 'max_depth': 10}. Best is trial 3 with value: 0.7768611509644

Best hyperparameters: {'n_estimators': 50, 'max_depth': 10}
Best accuracy: 0.7817391997449387


In [13]:
# get the best hyperparameters
best_params = study.best_params
# create a random forest classifier with the best hyperparameters
clf = RandomForestClassifier(**study.best_params,random_state=42)
clf.fit(X_train, y_train)

# make predictions on the test set
y_pred = clf.predict(X_test)

# calculate the accuracy of the model
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.7467532467532467


In [14]:
# optuna visualization
from optuna.visualization import plot_contour
from optuna.visualization import plot_edf
from optuna.visualization import plot_intermediate_values
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate

In [17]:
plot_optimization_history(study).show()

In [18]:
# parallel co-ordinate
plot_parallel_coordinate(study).show()

In [21]:
# plot_contour plot
plot_contour(study).show()

In [22]:
plt = plot_intermediate_values(study)
plt.show()

[W 2026-08-16 15:46:07,361] You need to set up the pruning feature to utilize `plot_intermediate_values()`


**Optimizing multiple Ml Model**

In [25]:
# import model
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier



In [37]:
# define obejective function
def objective2(trial):
  # chosse classifier algorithm to tune
  classifier_name = trial.suggest_categorical('classifier', ['RandomForest' , 'SVM', 'GradientBoosting'])
  if classifier_name == 'RandomForest':
    n_estimators = trial.suggest_int('n_estimators', 50,100)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,random_state=42)
  elif classifier_name ==  'SVM':
    C = trial.suggest_float('C', 1e-10, 1e10, log=True)
    model = SVC(C=C, random_state=42)
  elif classifier_name == 'GradientBoosting':
    n_estimators = trial.suggest_int('n_estimators', 50,100)
    learning_rate = trial.suggest_float('learning_rate', 1e-10, 1e10, log=True)
    model = GradientBoostingClassifier(n_estimators=n_estimators, learning_rate=learning_rate, random_state=42)
  score=cross_val_score(model,X_train,y_train,cv=3,scoring='accuracy').mean()
  return score





In [38]:
# create a study and optimize it using CmaEsSample
study=optuna.create_study(direction='maximize')
study.optimize(objective2,n_trials=100)

[I 2026-08-16 15:57:36,501] A new study created in memory with name: no-name-8d5e3e97-f08a-4ddf-b1cc-47fb85aa5391
[I 2026-08-16 15:57:37,160] Trial 0 finished with value: 0.7540411286465806 and parameters: {'classifier': 'RandomForest', 'n_estimators': 68, 'max_depth': 4}. Best is trial 0 with value: 0.7540411286465806.
[I 2026-08-16 15:57:37,230] Trial 1 finished with value: 0.6530926191614858 and parameters: {'classifier': 'SVM', 'C': 4.206579656307799e-10}. Best is trial 0 with value: 0.7540411286465806.
[I 2026-08-16 15:57:38,004] Trial 2 finished with value: 0.2931452255699028 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 74, 'learning_rate': 25.980655161379726}. Best is trial 0 with value: 0.7540411286465806.
[I 2026-08-16 15:57:38,070] Trial 3 finished with value: 0.6530926191614858 and parameters: {'classifier': 'SVM', 'C': 5.613832343873673e-06}. Best is trial 0 with value: 0.7540411286465806.
[I 2026-08-16 15:57:38,878] Trial 4 finished with value: 0.7654

In [42]:
best_trail=study.best_trial
print(best_trail.params)
print(best_trail.value)

{'classifier': 'RandomForest', 'n_estimators': 50, 'max_depth': 10}
0.7817391997449387
